# 🚀 Master Notebook: Huấn Luyện Toàn Bộ 5 Agent Trên Kaggle Dual T4 (`2 × 16GB = 32GB VRAM`)
**Đã tích hợp sẵn Google Drive Folder: [`1upmXzTIlxpcCVGU8Vp9ZGV3xNa1yu-Q6`](https://drive.google.com/drive/folders/1upmXzTIlxpcCVGU8Vp9ZGV3xNa1yu-Q6?usp=sharing)**

Notebook này gọi lần lượt gói kiến trúc `src/` độc lập của cả 5 Agent:
1. [`training_lab/agent1_retrieval/train_agent1_t4x2.py`](https://github.com/ducmanh-jr/Multi-Agent_RAG/tree/main/training_lab/agent1_retrieval) — **Agent 1: Retrieval Router** (`Qwen2.5-7B` + DoRA + rsLoRA `r=64` + SDPA)
2. [`training_lab/agent2_reader/train_agent2_t4x2.py`](https://github.com/ducmanh-jr/Multi-Agent_RAG/tree/main/training_lab/agent2_reader) — **Agent 2: Document Layout Reader** (`LayoutLMv3` 11-Class + Multi-Class Focal Loss $\gamma=2.0$)
3. [`training_lab/agent3_summarizer/train_agent3_t4x2.py`](https://github.com/ducmanh-jr/Multi-Agent_RAG/tree/main/training_lab/agent3_summarizer) — **Agent 3: Academic Summarizer** (`Generator` SFT+RPO/DPO & `Critic` Hard-Negatives)
4. [`training_lab/agent4_fact_checker/train_agent4_t4x2.py`](https://github.com/ducmanh-jr/Multi-Agent_RAG/tree/main/training_lab/agent4_fact_checker) — **Agent 4: Fact-Checker NLI Verifier** (`BGE-Reranker-v2-M3` 3-Class + Focal Loss + R-Drop + Temperature Calibration)
5. [`training_lab/agent5_writer/train_agent5_t4x2.py`](https://github.com/ducmanh-jr/Multi-Agent_RAG/tree/main/training_lab/agent5_writer) — **Agent 5: Academic Report Writer** (`Qwen2.5-7B` Two-Stage SFT + RPO/DPO Citation Alignment)

In [ ]:
# BƯỚC 1: CẤU HÌNH GOOGLE DRIVE FOLDER (ĐÃ ĐIỀN SẴN TỪ .ENV) & CHỌN AGENT CẦN HUẤN LUYỆN
GOOGLE_DRIVE_FOLDER_LINK = "https://drive.google.com/drive/folders/1upmXzTIlxpcCVGU8Vp9ZGV3xNa1yu-Q6?usp=sharing"
GOOGLE_DRIVE_FOLDER_ID = "1upmXzTIlxpcCVGU8Vp9ZGV3xNa1yu-Q6"
COLAB_DRIVE_SAVE_PATH = "/content/drive/MyDrive/drive"
GITHUB_TOKEN = ""  # (Tùy chọn) Điền GitHub Personal Access Token nếu muốn tự động git push trọng số mới về thẳng GitHub

RUN_AGENT_1 = True   # Agent 1: Retrieval Router & Query Planner (DoRA + rsLoRA r=64)
RUN_AGENT_2 = True   # Agent 2: Document Reader (LayoutLMv3 11-Class Focal Loss)
RUN_AGENT_3 = True   # Agent 3: Summarizer (Generator SFT+RPO/DPO & Critic Hard-Negatives)
RUN_AGENT_4 = True   # Agent 4: Fact-Checker NLI Verifier (Focal Loss + R-Drop + ECE Calibration)
RUN_AGENT_5 = True   # Agent 5: Academic Report Writer (Two-Stage SFT + RPO/DPO Alignment)

In [ ]:
# BƯỚC 2: CLONE GITHUB, CÀI ĐẶT THƯ VIỆN & ĐỒNG BỘ TRỌNG SỐ TỪ GOOGLE DRIVE
# Khac phuc triet de loi C++ operator torchvision::nms does not exist:
!pip uninstall -y -q torchvision
!pip install -q transformers peft trl bitsandbytes datasets accelerate sentencepiece gdown safetensors scikit-learn PyDrive2
import os, subprocess, torch, gdown

print(f'[GPU Check] Số lượng GPU khả dụng: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  -> GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB VRAM)')

PROJECT_DIR = '/kaggle/working/Multi-Agent_RAG'
if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/ducmanh-jr/Multi-Agent_RAG.git', PROJECT_DIR], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=False)
    subprocess.run(['git', '-C', PROJECT_DIR, 'reset', '--hard', 'origin/main'], check=False)
os.chdir(PROJECT_DIR)

# Tự động kiểm tra và tải các file có sẵn trong thư mục Google Drive của bạn về máy ảo:
DRIVE_SYNC_DIR = os.path.join(PROJECT_DIR, 'kaggle', 'drive')
os.makedirs(DRIVE_SYNC_DIR, exist_ok=True)
try:
    print(f'📥 Đang kiểm tra dữ liệu từ Google Drive Folder ({GOOGLE_DRIVE_FOLDER_ID})...')
    gdown.download_folder(url=GOOGLE_DRIVE_FOLDER_LINK, output=DRIVE_SYNC_DIR, quiet=True, remaining_ok=True)
    print('✅ Đã kết nối thành công với Google Drive Folder:', GOOGLE_DRIVE_FOLDER_LINK)
except Exception as e:
    print('ℹ️ Thư mục Google Drive hiện đang trống hoặc sẵn sàng nhận trọng số mới:', e)

In [ ]:
# BƯỚC 3: CHẠY HUẤN LUYỆN TUẦN TỰ TỪNG AGENT (MỖI AGENT CHẠY TRONG TIẾN TRÌNH RIÊNG ĐỂ XẢ 100% VRAM)
if RUN_AGENT_1:
    print('\n' + '='*80 + '\n🧭 [1/5] HUẤN LUYỆN AGENT 1: RETRIEVAL ROUTER (DoRA + rsLoRA r=64)\n' + '='*80)
    !CUDA_VISIBLE_DEVICES=0 python training_lab/agent1_retrieval/train_agent1_t4x2.py --epochs 3

if RUN_AGENT_2:
    print('\n' + '='*80 + '\n📄 [2/5] HUẤN LUYỆN AGENT 2: DOCUMENT READER (LayoutLMv3 Focal Loss)\n' + '='*80)
    !CUDA_VISIBLE_DEVICES=0 python training_lab/agent2_reader/train_agent2_t4x2.py --epochs 5

if RUN_AGENT_3:
    print('\n' + '='*80 + '\n✍️ [3/5] HUẤN LUYỆN AGENT 3: SUMMARIZER (Generator SFT+RPO/DPO & Critic)\n' + '='*80)
    !CUDA_VISIBLE_DEVICES=0 python training_lab/agent3_summarizer/train_agent3_t4x2.py --role both --epochs-sft 3 --epochs-dpo 2

if RUN_AGENT_4:
    print('\n' + '='*80 + '\n🛡️ [4/5] HUẤN LUYỆN AGENT 4: FACT-CHECKER (Cross-Encoder Focal + R-Drop)\n' + '='*80)
    !CUDA_VISIBLE_DEVICES=0 python training_lab/agent4_fact_checker/train_agent4_t4x2.py --epochs 4

if RUN_AGENT_5:
    print('\n' + '='*80 + '\n📝 [5/5] HUẤN LUYỆN AGENT 5: ACADEMIC WRITER (Two-Stage SFT + RPO/DPO)\n' + '='*80)
    !CUDA_VISIBLE_DEVICES=0 python training_lab/agent5_writer/train_agent5_t4x2.py --epochs-sft 3 --epochs-dpo 2

In [ ]:
# BƯỚC 4: TỰ ĐỘNG ĐÓNG GÓI & LƯU TRỌNG SỐ VỀ GOOGLE DRIVE / GITHUB / KAGGLE OUTPUT
import os, shutil, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = os.path.exists('/var/colab/hostname') and not IS_KAGGLE

BUNDLE_DIR = Path(PROJECT_DIR) / 'kaggle' / 'drive' / 'MultiAgent_RAG_Weights'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

agent_weight_map = {
    'agent1_retrieval': Path(PROJECT_DIR) / 'training_lab/agent1_retrieval/weights',
    'agent2_reader': Path(PROJECT_DIR) / 'training_lab/agent2_reader/weights',
    'agent3_summarizer': Path(PROJECT_DIR) / 'training_lab/agent3_summarizer/weights',
    'agent4_fact_checker': Path(PROJECT_DIR) / 'training_lab/agent4_fact_checker/weights',
    'agent5_writer': Path(PROJECT_DIR) / 'training_lab/agent5_writer/weights',
}

for name, w_path in agent_weight_map.items():
    if w_path.exists():
        dest = BUNDLE_DIR / name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(w_path, dest, ignore=shutil.ignore_patterns('*checkpoint*'))
        print(f'📦 Đã gom trọng số {name} -> {dest}')

OUT_ZIP = '/kaggle/working/MultiAgent_RAG_Weights_All_5_Agents.zip' if IS_KAGGLE else '/content/MultiAgent_RAG_Weights_All_5_Agents.zip'
shutil.make_archive(OUT_ZIP.replace('.zip', ''), 'zip', str(BUNDLE_DIR))
print(f'✅ Đã đóng gói toàn bộ trọng số 5 Agent tại: {OUT_ZIP}')

# 1. Nếu chạy trên Kaggle: File zip đã sẵn sàng ở tab Output bên phải màn hình
if IS_KAGGLE:
    print('=' * 80)
    print(f'🎉 [KAGGLE THÀNH CÔNG] File trọng số đã được lưu tại: {OUT_ZIP}')
    print('👉 Bạn có thể tải trực tiếp file zip từ tab Output bên phải màn hình Kaggle!')
    print('=' * 80)

# 2. Nếu chạy trên Google Colab: Mount drive an toàn và copy vào thư mục MyDrive/drive
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(COLAB_DRIVE_SAVE_PATH, exist_ok=True)
        shutil.copy2(OUT_ZIP, os.path.join(COLAB_DRIVE_SAVE_PATH, 'MultiAgent_RAG_Weights_All_5_Agents.zip'))
        shutil.copytree(str(BUNDLE_DIR), os.path.join(COLAB_DRIVE_SAVE_PATH, 'MultiAgent_RAG_Weights'), dirs_exist_ok=True)
        print(f'🎉 ĐÃ LƯU TRỰC TIẾP VÀO GOOGLE DRIVE ({GOOGLE_DRIVE_FOLDER_LINK}) TẠI: {COLAB_DRIVE_SAVE_PATH}')
    except Exception as e:
        print('⚠️ Không thể mount Google Drive trên môi trường này:', e)

# 3. Nếu có GITHUB_TOKEN: Tự động commit & push trọng số FP16 (<80MB) lên thẳng GitHub
if GITHUB_TOKEN:
    subprocess.run(['git', 'config', 'user.email', 'kaggle-t4x2@multiagent-rag.ai'], check=False)
    subprocess.run(['git', 'config', 'user.name', 'Kaggle T4x2 Trainer'], check=False)
    subprocess.run(['git', 'add', 'training_lab/*/weights'], check=False)
    subprocess.run(['git', 'commit', '-m', 'chore(weights): update fine-tuned FP16 adapters from Kaggle T4x2'], check=False)
    subprocess.run(['git', 'push', f'https://{GITHUB_TOKEN}@github.com/ducmanh-jr/Multi-Agent_RAG.git', 'HEAD:main'], check=False)
    print('🎉 ĐÃ TỰ ĐỘNG PUSH TRỌNG SỐ MỚI LÊN GITHUB REPOSITORY!')
